# Notebook 2 — Inspect and profile the Amazon Reviews 2023 `Video_Games` dataset

This notebook is the **inspection and profiling** step of the project pipeline.

It is designed to:

- load the local raw files created in **Notebook 1**
- inspect the structure of the **reviews** and **metadata** datasets with Spark DataFrames
- profile key fields such as ratings, timestamps, helpful votes, verified purchase, price, and product metadata
- test which join keys look strongest for later Spark + MongoDB work
- identify **analysis directions** that match the proposal and grading criteria

Before building the full pipeline, we should understand the dataset structure, data quality, and join strategy clearly.

## Assumptions

1. **Notebook 1 has already been run successfully** and created local raw files.
2. The raw review file and metadata file are stored inside the repository under `data/raw/amazon_reviews_2023/video_games/`.
3. We are still working with the **raw** local JSON Lines files, so this notebook focuses on inspection and light profiling rather than full cleaning.
4. Because these datasets are large, we only convert **small aggregated results** to Pandas for plotting.

## 1. Imports

In [1]:
from pathlib import Path
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## 2. Define file paths

In [2]:
reviews_path = Path("../data/raw/amazon_reviews_2023/video_games/reviews.jsonl")
metadata_path = Path("../data/raw/amazon_reviews_2023/video_games/metadata.jsonl")

print("Reviews path:", reviews_path.resolve())
print("Metadata path:", metadata_path.resolve())

Reviews path: C:\Users\olive\git\HAMK\BigDataAnalytics\Project\big-data-video-games-reviews\data\raw\amazon_reviews_2023\video_games\reviews.jsonl
Metadata path: C:\Users\olive\git\HAMK\BigDataAnalytics\Project\big-data-video-games-reviews\data\raw\amazon_reviews_2023\video_games\metadata.jsonl


## 3. Basic file checks

In [3]:
assert reviews_path.exists(), f"Missing reviews file: {reviews_path}"
assert metadata_path.exists(), f"Missing metadata file: {metadata_path}"

print("Review file exists:", reviews_path.exists())
print("Metadata file exists:", metadata_path.exists())

Review file exists: True
Metadata file exists: True


## 4. Start Spark and load the raw JSON files

This cell starts a Spark session and loads the raw review and metadata files into Spark DataFrames.

In [4]:
spark = (
    SparkSession.builder
    .appName("Notebook2_Inspect_Profile_Amazon_Video_Games")
    .getOrCreate()
)

reviews_df = spark.read.json(str(reviews_path))
metadata_df = spark.read.json(str(metadata_path))

reviews_count = reviews_df.count()
metadata_count = metadata_df.count()

print(f"Reviews rows:  {reviews_count:,}")
print(f"Metadata rows: {metadata_count:,}")

Reviews rows:  4,624,615
Metadata rows: 137,269


## 5. Inspect schema and columns

Before cleaning, joining, or planning the MongoDB structure, we first inspect the schema of both datasets.

This helps us see which fields are available, what data types Spark assigned, and whether any columns contain nested structures.

In [5]:
print("=== Reviews schema ===")
reviews_df.printSchema()

print("\n=== Metadata schema ===")
metadata_df.printSchema()

=== Reviews schema ===
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)


=== Metadata schema ===
root
 |-- author: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |   

In [6]:
print("Review columns:")
print(reviews_df.columns)

print("\nMetadata columns:")
print(metadata_df.columns)

Review columns:
['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']

Metadata columns:
['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']


## 6. Preview example rows

After checking the schema, we should look at a few actual rows from both datasets.

This helps us understand:
- what the raw values look like in practice
- whether key text fields are readable
- how metadata fields such as categories appear

In [7]:
reviews_df.select(
    "parent_asin",
    "asin",
    "rating",
    "helpful_vote",
    "verified_purchase",
    "timestamp",
    "title",
    "text"
).show(5, truncate=100)

metadata_df.select(
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "categories"
).show(5, truncate=100)

+-----------+----------+------+------------+-----------------+-------------+------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|parent_asin|      asin|rating|helpful_vote|verified_purchase|    timestamp|                                                                   title|                                                                                                text|
+-----------+----------+------+------------+-----------------+-------------+------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
| B07DK1H3H5|B07DJWBYKP|   4.0|           0|             true|1608186804795|                                          It’s pretty sexual. Not my fav|I’m playing on ps5 and it’s interesting.  It’s unique, massive, and has a neat story.  People are.

## 7. Check missing values in key columns

Now we do a simple missing-value check for the most important columns only.

The goal is not full data cleaning yet. We just want a basic understanding of data quality in the reviews and metadata datasets.

In [8]:
def missing_summary(df, columns, total_rows):
    summary = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columns]
    ).toPandas().T.reset_index()

    summary.columns = ["column", "null_count"]
    summary["null_pct"] = (summary["null_count"] / total_rows * 100).round(2)

    return summary.sort_values("null_pct", ascending=False).reset_index(drop=True)

In [9]:
review_cols_to_check = [
    "asin",
    "parent_asin",
    "rating",
    "text",
    "timestamp",
    "title",
    "user_id",
    "verified_purchase",
    "helpful_vote"
]

metadata_cols_to_check = [
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "categories"
]

reviews_missing = missing_summary(reviews_df, review_cols_to_check, reviews_count)
metadata_missing = missing_summary(metadata_df, metadata_cols_to_check, metadata_count)

print("=== Missing values in reviews ===")
display(reviews_missing)

print("=== Missing values in metadata ===")
display(metadata_missing)

=== Missing values in reviews ===


,column,null_count,null_pct
0,asin,0,0.0
1,parent_asin,0,0.0
2,rating,0,0.0
3,text,0,0.0
4,timestamp,0,0.0
5,title,0,0.0
6,user_id,0,0.0
7,verified_purchase,0,0.0
8,helpful_vote,0,0.0


=== Missing values in metadata ===


,column,null_count,null_pct
0,main_category,11035,8.04
1,store,4361,3.18
2,title,0,0.00
3,parent_asin,0,0.00
4,average_rating,0,0.00
5,rating_number,0,0.00
6,price,0,0.00
7,categories,0,0.00


## 8. Basic profile of the reviews dataset

Now we create a few simple derived columns to better understand the reviews data:

- `review_time` converts the raw Unix timestamp into a readable timestamp
- `review_year` helps us inspect how reviews are distributed over time
- `text_length` gives a simple measure of review length

Then we calculate a few high-level summary statistics.

In [10]:
reviews_profile_df = (
    reviews_df
    .withColumn(
        "review_time",
        F.to_timestamp(F.from_unixtime((F.col("timestamp") / 1000).cast("bigint")))
    )
    .withColumn("review_year", F.year("review_time"))
    .withColumn("text_length", F.length(F.coalesce(F.col("text"), F.lit(""))))
)

In [11]:
reviews_profile_df.select(
    F.count("*").alias("rows"),
    F.approx_count_distinct("user_id").alias("approx_distinct_users"),
    F.approx_count_distinct("asin").alias("approx_distinct_asins"),
    F.approx_count_distinct("parent_asin").alias("approx_distinct_parent_asins"),
    F.avg("rating").alias("avg_rating"),
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
    F.avg("text_length").alias("avg_text_length")
).show(truncate=False)

+-------+---------------------+---------------------+----------------------------+----------------+----------+----------+------------------+
|rows   |approx_distinct_users|approx_distinct_asins|approx_distinct_parent_asins|avg_rating      |min_rating|max_rating|avg_text_length   |
+-------+---------------------+---------------------+----------------------------+----------------+----------+----------+------------------+
|4624615|2616785              |168080               |132156                      |4.04745951825179|1.0       |5.0       |307.59483740808696|
+-------+---------------------+---------------------+----------------------------+----------------+----------+----------+------------------+



### 8.1 Rating distribution

Now we inspect how ratings are distributed.

This gives us a first view of customer sentiment in the dataset and helps us see whether the ratings are balanced or concentrated toward higher scores.

In [12]:
rating_dist = (
    reviews_profile_df
    .groupBy("rating")
    .count()
    .orderBy("rating")
)

rating_dist.show()

+------+-------+
|rating|  count|
+------+-------+
|   1.0| 589519|
|   2.0| 249878|
|   3.0| 340086|
|   4.0| 617251|
|   5.0|2827881|
+------+-------+



### 8.2 Verified purchase distribution

Now we check how reviews are split between verified and non-verified purchases.

This helps us understand whether most reviews come from confirmed buyers and whether verified purchase could be a useful analysis variable later.

In [13]:
verified_dist = (
    reviews_profile_df
    .groupBy("verified_purchase")
    .count()
    .orderBy("verified_purchase")
)

verified_dist.show()

+-----------------+-------+
|verified_purchase|  count|
+-----------------+-------+
|            false| 641808|
|             true|3982807|
+-----------------+-------+



### 8.3 Helpful vote profile

Now we inspect the helpful vote field.

This helps us see whether helpful votes are common, how large they usually are, and whether the variable looks useful for later analysis.

In [14]:
reviews_profile_df.select(
    F.avg("helpful_vote").alias("avg_helpful_vote"),
    F.max("helpful_vote").alias("max_helpful_vote"),
    F.expr("percentile_approx(helpful_vote, array(0.5, 0.9, 0.99))").alias("helpful_vote_percentiles")
).show(truncate=False)

+-----------------+----------------+------------------------+
|avg_helpful_vote |max_helpful_vote|helpful_vote_percentiles|
+-----------------+----------------+------------------------+
|1.227983302393821|10369           |[0, 2, 18]              |
+-----------------+----------------+------------------------+



In [15]:
helpful_nonzero = reviews_profile_df.filter(F.col("helpful_vote") > 0).count()

print(f"Reviews with helpful_vote > 0: {helpful_nonzero:,}")
print(f"Share of reviews with helpful_vote > 0: {helpful_nonzero / reviews_count:.2%}")

Reviews with helpful_vote > 0: 1,167,726
Share of reviews with helpful_vote > 0: 25.25%


### 8.4 Reviews over time by year

Now we inspect how the number of reviews changes over time.

For Notebook 2, using the year level is enough. It gives a basic understanding of the dataset timeline.

In [16]:
reviews_by_year = (
    reviews_profile_df
    .filter(F.col("review_year").isNotNull())
    .groupBy("review_year")
    .count()
    .orderBy("review_year")
)

reviews_by_year.show(30)

+-----------+------+
|review_year| count|
+-----------+------+
|       1998|     6|
|       1999|   455|
|       2000|  4821|
|       2001| 10836|
|       2002| 14345|
|       2003| 14133|
|       2004| 15268|
|       2005| 16679|
|       2006| 16468|
|       2007| 28761|
|       2008| 39511|
|       2009| 50904|
|       2010| 64460|
|       2011| 90317|
|       2012|112039|
|       2013|249462|
|       2014|323445|
|       2015|410524|
|       2016|390293|
|       2017|364240|
|       2018|379482|
|       2019|447238|
|       2020|521460|
|       2021|479609|
|       2022|399136|
|       2023|180723|
+-----------+------+



## 9. Basic profile of the metadata dataset

The metadata dataset is more nested than the reviews dataset.

We create a few simple helper columns:
- `price_numeric` to see whether price can be used later
- `n_categories` to see how many category labels products usually have

In [17]:
metadata_profile_df = (
    metadata_df
    .withColumn(
        "price_clean",
        F.regexp_replace(F.col("price"), "[$,]", "")
    )
    .withColumn(
        "price_numeric",
        F.when(
            F.col("price_clean").rlike(r"^[0-9]+(\.[0-9]+)?$"),
            F.col("price_clean").cast("double")
        )
    )
    .withColumn(
        "n_categories",
        F.when(F.col("categories").isNull(), F.lit(0)).otherwise(F.size("categories"))
    )
)

### 9.1 Metadata summary statistics

Now we calculate a few high-level summary statistics for the metadata dataset.

In [18]:
metadata_profile_df.select(
    F.count("*").alias("rows"),
    F.approx_count_distinct("parent_asin").alias("approx_distinct_parent_asins"),
    F.approx_count_distinct("store").alias("approx_distinct_stores"),
    F.avg("average_rating").alias("avg_average_rating"),
    F.avg("rating_number").alias("avg_rating_number"),
    F.avg("n_categories").alias("avg_categories_per_product")
).show(truncate=False)

+------+----------------------------+----------------------+------------------+-----------------+--------------------------+
|rows  |approx_distinct_parent_asins|approx_distinct_stores|avg_average_rating|avg_rating_number|avg_categories_per_product|
+------+----------------------------+----------------------+------------------+-----------------+--------------------------+
|137269|132156                      |17858                 |3.9952960974436973|244.3050870917687|3.8923864820170615        |
+------+----------------------------+----------------------+------------------+-----------------+--------------------------+



### 9.2 Main category distribution

Now we inspect the `main_category` field.

This helps us check whether the metadata is consistently labeled as `Video Games` and how many rows are missing this field.

In [19]:
metadata_profile_df.groupBy("main_category").count().orderBy(F.desc("count")).show(truncate=False)

+----------------------------+-----+
|main_category               |count|
+----------------------------+-----+
|Video Games                 |81255|
|Computers                   |17235|
|All Electronics             |14816|
|NULL                        |11035|
|Cell Phones & Accessories   |3884 |
|Toys & Games                |2733 |
|Software                    |1511 |
|Industrial & Scientific     |1079 |
|Amazon Home                 |737  |
|Home Audio & Theater        |443  |
|Tools & Home Improvement    |369  |
|Office Products             |295  |
|Sports & Outdoors           |244  |
|Buy a Kindle                |220  |
|Movies & TV                 |197  |
|Books                       |196  |
|Musical Instruments         |154  |
|All Beauty                  |126  |
|Camera & Photo              |117  |
|Portable Audio & Accessories|112  |
+----------------------------+-----+
only showing top 20 rows


### 9.3 Price summary

Now we inspect whether the `price` field looks usable for later analysis.

Because `price` was stored as text in the raw metadata, we created `price_numeric` in the previous step. Here we check how many rows could be converted into numeric values and what the basic price distribution looks like.

In [20]:
metadata_profile_df.select(
    F.sum(F.col("price_numeric").isNotNull().cast("int")).alias("rows_with_numeric_price"),
    F.avg("price_numeric").alias("avg_price"),
    F.min("price_numeric").alias("min_price"),
    F.max("price_numeric").alias("max_price"),
    F.expr("percentile_approx(price_numeric, array(0.25, 0.5, 0.75))").alias("price_quartiles")
).show(truncate=False)

+-----------------------+------------------+---------+---------+--------------------+
|rows_with_numeric_price|avg_price         |min_price|max_price|price_quartiles     |
+-----------------------+------------------+---------+---------+--------------------+
|61972                  |45.717351868585645|0.0      |3499.99  |[13.0, 24.95, 45.99]|
+-----------------------+------------------+---------+---------+--------------------+



### 9.4 Top values in the categories array

The `main_category` field was not cleanly limited to only `Video Games`, so now we inspect the `categories` array.

This helps us see whether the detailed category labels are more useful for understanding the products.

In [21]:
(
    metadata_profile_df
    .select(F.explode_outer("categories").alias("category"))
    .filter(F.col("category").isNotNull())
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

+------------------------------+------+
|category                      |count |
+------------------------------+------+
|Video Games                   |124619|
|Accessories                   |63120 |
|Legacy Systems                |50925 |
|Games                         |46533 |
|PC                            |37500 |
|Nintendo Systems              |23525 |
|PlayStation Systems           |14987 |
|Nintendo Switch               |12352 |
|Controllers                   |10875 |
|PlayStation 4                 |10145 |
|Xbox Systems                  |8282  |
|Headsets                      |7222  |
|Xbox One                      |6916  |
|Faceplates, Protectors & Skins|6776  |
|Cases & Storage               |6087  |
|Wii                           |5734  |
|Xbox 360                      |5430  |
|PlayStation 3                 |4844  |
|Gaming Keyboards              |4814  |
|Nintendo DS                   |4295  |
+------------------------------+------+
only showing top 20 rows


## 10. Check the main join key

The most important structural question now is whether `parent_asin` works well as a join key between reviews and metadata.

We start by checking:
- how many distinct `parent_asin` values appear in reviews
- how many appear in metadata
- how many overlap

In [22]:
review_keys = (
    reviews_df
    .select("parent_asin")
    .filter(F.col("parent_asin").isNotNull())
    .distinct()
)

metadata_keys = (
    metadata_df
    .select("parent_asin")
    .filter(F.col("parent_asin").isNotNull())
    .distinct()
)

review_key_count = review_keys.count()
metadata_key_count = metadata_keys.count()
matching_key_count = review_keys.join(metadata_keys, on="parent_asin", how="inner").count()

print(f"Distinct parent_asin values in reviews:  {review_key_count:,}")
print(f"Distinct parent_asin values in metadata: {metadata_key_count:,}")
print(f"Matching parent_asin values:             {matching_key_count:,}")

Distinct parent_asin values in reviews:  137,249
Distinct parent_asin values in metadata: 137,269
Matching parent_asin values:             137,249


### 10.1 Review-row match coverage

Now we check how many review rows have matching metadata through `parent_asin`.

This tells us whether the join works well not only for distinct keys, but also for the full reviews dataset.

In [23]:
reviews_with_matching_metadata = (
    reviews_df
    .join(metadata_keys, on="parent_asin", how="left_semi")
    .count()
)

print(f"Review rows with matching metadata: {reviews_with_matching_metadata:,}")
print(f"Share of reviews with matching metadata: {reviews_with_matching_metadata / reviews_count:.2%}")

Review rows with matching metadata: 4,624,615
Share of reviews with matching metadata: 100.00%


### 10.2 Metadata keys without matching reviews

Now we check whether there are metadata products that do not appear in the reviews dataset.

This helps us understand the small difference between the metadata key count and the review key count.

In [24]:
metadata_without_reviews = metadata_keys.join(
    review_keys,
    on="parent_asin",
    how="left_anti"
)

print(f"Metadata keys with no matching reviews: {metadata_without_reviews.count():,}")

metadata_without_reviews.show(10, truncate=False)

Metadata keys with no matching reviews: 20
+-----------+
|parent_asin|
+-----------+
|B00005B70O |
|B0000ZPTBK |
|B0000ZPUB4 |
|B07D45SYHT |
|B081Y55VHP |
|B08NWJZK77 |
|B07F73J824 |
|B07H7RVP91 |
|B0000ZPSRU |
|B07MJC4TDD |
+-----------+
only showing top 10 rows


## 11. Key findings from Notebook 2

This notebook gave us a basic understanding of the Amazon Reviews 2023 `Video_Games` dataset.

### Main findings

- The dataset is large enough for a Big Data project:
  - **4,624,615 review rows**
  - **137,269 metadata rows**

- The **reviews dataset is very clean** in the main columns used here:
  - no missing values in `asin`, `parent_asin`, `rating`, `text`, `timestamp`, `title`, `user_id`, `verified_purchase`, or `helpful_vote`

- The review data is strongly **skewed toward high ratings**:
  - the average rating is about **4.05**
  - **5-star reviews** are by far the most common

- Most reviews come from **verified purchases**:
  - about **3.98 million** verified reviews
  - about **0.64 million** non-verified reviews

- The `helpful_vote` field is usable, but highly skewed:
  - many reviews have **0 helpful votes**
  - a smaller number of reviews have much higher values

- The review timeline spans from **1998 to 2023**, with much higher review volume in later years

- The metadata dataset contains useful product information such as:
  - `title`
  - `average_rating`
  - `rating_number`
  - `price`
  - `store`
  - `categories`

- The metadata is a bit less clean than the reviews data:
  - `main_category` and `store` contain some missing values
  - `price` is only partly available as a usable numeric field

- The `main_category` field is **not reliable enough on its own** for defining the Video Games subset, because many rows are labeled with other categories such as `Computers` and `All Electronics`

- The `categories` array looks more useful than `main_category`, because it contains detailed labels such as `Video Games`, `Accessories`, `Legacy Systems`, `PC`, and platform-specific categories

- `parent_asin` is a **very strong join key** between reviews and metadata:
  - **137,249** distinct `parent_asin` values appear in reviews
  - **137,269** distinct `parent_asin` values appear in metadata
  - **137,249** keys overlap
  - **100% of review rows** have matching metadata
  - only **20 metadata keys** have no matching review rows

### Conclusion

The dataset looks well suited for the project.  
The reviews data is large and clean, the metadata is rich enough to support product-level analysis, and `parent_asin` provides a reliable way to join the two datasets.

The next notebook should focus on light cleaning and preparing a joined analytical dataset for further Spark analysis and later MongoDB schema design.